# GaussianSplat — train on Colab

Upload the `session-<id>.zip` produced by the app, train a 3D Gaussian Splatting scene on Colab's GPU, then download the resulting `output.splat` and load it back into the in-app viewer.

**Runtime:** before running, set **Runtime → Change runtime type → T4 GPU** (free tier is enough).

In [ ]:
# 1. Sanity-check GPU
!nvidia-smi

In [ ]:
# 2. Install nerfstudio (pinned) + gsplat. ~5 min on a cold runtime.
#    Dependency-conflict warnings during install are noise from Colab's
#    pre-installed packages — they don't affect training.
%pip install --quiet 'nerfstudio==1.1.5' gsplat

In [ ]:
# 3. Upload your session-<id>.zip from the phone (AirDropped to your Mac first).
from google.colab import files
import os, zipfile, shutil, json, math
from PIL import Image

uploaded = files.upload()
if not uploaded:
    raise SystemExit('No file uploaded.')
zip_name = next(iter(uploaded))
shutil.rmtree('session', ignore_errors=True)
os.makedirs('session', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('session')

tfm_path = 'session/transforms.json'
with open(tfm_path) as f:
    tfm = json.load(f)
imgs = sorted(p for p in os.listdir('session/images') if p.lower().endswith(('.jpg', '.png')))
n_imgs = len(imgs)
print(f'transforms.json keys: {sorted(tfm.keys())}')
print(f'frames in transforms.json: {len(tfm.get("frames", []))}')
print(f'images on disk: {n_imgs}')
assert n_imgs > 0, 'No images extracted — bad zip?'
assert len(tfm.get('frames', [])) == n_imgs, 'frame count mismatch — re-export from the app.'

# Patch intrinsics to match the actual image dimensions. nerfstudio asserts
# on a mismatch and crashes data caching, which is the common failure mode
# when transforms.json was synthesised on-device with guessed dims.
real_w, real_h = Image.open(f'session/images/{imgs[0]}').size
if (tfm.get('w'), tfm.get('h')) != (real_w, real_h):
    horizontal_fov_deg = 70.0
    focal = 0.5 * real_w / math.tan(math.radians(horizontal_fov_deg) / 2)
    tfm['w'] = real_w
    tfm['h'] = real_h
    tfm['cx'] = real_w / 2
    tfm['cy'] = real_h / 2
    tfm['fl_x'] = focal
    tfm['fl_y'] = focal
    with open(tfm_path, 'w') as f:
        json.dump(tfm, f, indent=2)
    print(f'patched intrinsics to {real_w}x{real_h}, fl={focal:.1f}')

In [ ]:
# 4. Train splatfacto.
#
# Key flags:
#   --vis tensorboard               headless; the websocket viewer can't bind a port in Colab.
#   --pipeline.model.random-init    seed Gaussians from random points instead of SfM, which we
#                                   don't have when poses came from the synthetic orbit.
#   --max-num-iterations 7000       ~10 min on T4. Bump to 30000 for paper-quality.
#
# subprocess.Popen + line-buffered iteration guarantees we see every log line
# even if the kernel dies — `!ns-train …` can swallow output on hard kills.
import subprocess, shutil

shutil.rmtree('runs', ignore_errors=True)

cmd = [
    'ns-train', 'splatfacto',
    '--data', 'session',
    '--output-dir', 'runs',
    '--vis', 'tensorboard',
    '--max-num-iterations', '7000',
    '--steps-per-save', '2000',
    '--pipeline.model.random-init', 'True',
    '--pipeline.model.num-random', '50000',
    'nerfstudio-data',
    '--orientation-method', 'none',
    '--center-method', 'none',
    '--auto-scale-poses', 'False',
]
print('>', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n--- ns-train exit code: {proc.returncode} ---')

In [ ]:
# 5. Find the trained .ply. If nothing was written, dump runs/ so we can see why.
import glob, os

plys = sorted(glob.glob('runs/**/point_cloud.ply', recursive=True))
if not plys:
    print('No point_cloud.ply produced. Contents of runs/:')
    for root, _, files in os.walk('runs'):
        for fn in files:
            full = os.path.join(root, fn)
            print(f'  {full}  ({os.path.getsize(full)} bytes)')
    raise SystemExit('Look up at cell 4 output for the real ns-train error.')
ply = plys[-1]
print(f'using ply: {ply}  ({os.path.getsize(ply)} bytes)')

In [ ]:
# 6. Convert the trained .ply to the .splat format the in-app viewer reads.
import subprocess, urllib.request, os

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/ArioMoniri/GaussianSplat/main/scripts/ply_to_splat.py',
    'ply_to_splat.py',
)
subprocess.check_call(['python', 'ply_to_splat.py', ply, 'output.splat'])
print('wrote output.splat:', os.path.getsize('output.splat'), 'bytes')

In [ ]:
# 7. Download to your Mac. AirDrop back to the phone, then in the app:
#    Train → Import trained splat.
from google.colab import files
files.download('output.splat')